# Capítulo 15 — Cuando el modelo miente

**Cuaderno interactivo de *La servilleta y el ordenador*.**

Cada sección reproduce una figura del capítulo. La gracia no es ejecutarlas: es **cambiar los parámetros y comprobar si ocurre lo que esperabas**.

> Antes de ejecutar cada celda, escribe en una línea qué esperas ver. Después mira si ocurrió. Y después, por qué.


In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd() / ''))
sys.path.insert(0, '../../../herramientas')
sys.path.insert(0, str(pathlib.Path.cwd().parents[1] / 'herramientas'))

import numpy as np
import matplotlib.pyplot as plt
from estilo_libro import C, use_style, rng, save

use_style()
%matplotlib inline

---

## Cuatro modelos que coinciden en los datos y divergen fuera.

Ajuste de exponencial, logística, ley de potencias y polinomio a los mismos 20
puntos, con las predicciones extendidas al doble del rango.

La figura responde: ¿qué parte de una extrapolación viene de los datos y qué
parte de la física que has supuesto?

Ejecutar:  python fig_extrapolacion.py

*(script original: `codigo/fig_extrapolacion.py`)*

**Antes de ejecutar, escribe aquí qué esperas ver:**

> 


In [ ]:
import pathlib
import sys

import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import curve_fit

from estilo_libro import C, rng, save, use_style  # noqa: E402

use_style()
r = rng(151)

# "Verdad": logística en fase inicial
K, R, N0 = 4000.0, 0.42, 12.0
t_datos = np.arange(0, 11, 0.5)   # sólo la fase inicial: ahí todos coinciden
verdad = K / (1 + (K / N0 - 1) * np.exp(-R * t_datos))
y = verdad * np.exp(r.normal(0, 0.08, t_datos.size))

MODELOS = [
    ("exponencial", lambda t, a, b: a * np.exp(b * t), [10, 0.4], C.red),
    ("logística", lambda t, k, rr, n0: k / (1 + (k / n0 - 1) * np.exp(-rr * t)),
     [3000, 0.4, 12], C.green),
    ("ley de potencias", lambda t, a, p: a * (t + 1)**p, [10, 2.5], C.ochre),
    ("polinomio grado 3", lambda t, a, b, c, d: a + b * t + c * t**2 + d * t**3,
     [10, 1, 1, 1], C.purple),
]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10.4, 4.2))
tt = np.linspace(0, 40, 400)

for nombre, f, p0, color in MODELOS:
    try:
        p, _ = curve_fit(f, t_datos, y, p0=p0, maxfev=60000)
    except Exception:
        continue
    ajuste = f(t_datos, *p)
    rms = np.sqrt(np.mean((np.log(y) - np.log(np.abs(ajuste) + 1e-9))**2))
    ax1.plot(t_datos, ajuste, color=color, lw=1.8, label=f"{nombre} (rms {rms:.3f})")
    ax2.semilogy(tt, np.abs(f(tt, *p)), color=color, lw=1.8, label=nombre)
    print(f"{nombre:20s} rms(log) en los datos = {rms:.4f}   "
          f"predicción en t=40: {f(40.0, *p):.3g}")

ax1.plot(t_datos, y, "o", color=C.ink, ms=5, label="datos")
ax1.set_xlabel("$t$"), ax1.set_ylabel("$N$")
ax1.set_title("Dentro del rango: tres se distinguen a duras penas")
ax1.legend(fontsize=7.6)

verdad_larga = K / (1 + (K / N0 - 1) * np.exp(-R * tt))
ax2.semilogy(tt, verdad_larga, "--", color=C.ink, lw=2.0, label="verdad")
ax2.axvspan(0, 10, color=C.grey, alpha=0.18)
ax2.text(0.6, 3e6, "rango medido", fontsize=8.6, color=C.ink)
ax2.set_xlabel("$t$"), ax2.set_ylabel("$N$")
ax2.set_title("Fuera del rango: cuatro órdenes y medio de diferencia")
ax2.legend(fontsize=7.6, loc="lower right")
ax2.set_ylim(1, 1e9)

plt.show()

**¿Ocurrió lo que esperabas? ¿Por qué?**

> 

**Juega:** cambia un parámetro de la celda anterior, vuelve a ejecutarla y anota el efecto.

> 


---

## Sensibilidad local frente a global: la derivada parcial que engaña.

Un modelo no lineal donde la derivada parcial en el punto nominal dice que un
parámetro no importa, y el análisis global dice lo contrario.

La figura responde: ¿basta con mover un parámetro cada vez?

Ejecutar:  python fig_sensibilidad.py

*(script original: `codigo/fig_sensibilidad.py`)*

**Antes de ejecutar, escribe aquí qué esperas ver:**

> 


In [ ]:
import pathlib
import sys

import matplotlib.pyplot as plt
import numpy as np

from estilo_libro import C, rng, save, use_style  # noqa: E402

use_style()
r = rng(155)


def modelo(a, b):
    """Salida con interacción fuerte entre a y b."""
    return np.sin(a) * np.sin(b) + 0.3 * a


A0, B0 = np.pi / 2, np.pi / 2       # punto nominal: derivada de b nula

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10.4, 4.2))

# --- Sensibilidad local (un parámetro cada vez) --------------------------
d = np.linspace(-1.5, 1.5, 300)
ax1.plot(d, modelo(A0 + d, B0), color=C.blue, lw=2.0, label="variando $a$")
ax1.plot(d, modelo(A0, B0 + d), color=C.red, lw=2.0, label="variando $b$")
ax1.plot(0, modelo(A0, B0), "o", color=C.ink, ms=7)
ax1.set_xlabel("desviación desde el punto nominal")
ax1.set_ylabel("salida del modelo")
ax1.set_title("Local: «$b$ no influye, su derivada es cero»")
ax1.legend(fontsize=8.5)
ax1.annotate("plana en el nominal", xy=(0, modelo(A0, B0)),
             xytext=(-1.35, 1.05), fontsize=8.4, color=C.red,
             arrowprops=dict(arrowstyle="->", color=C.red, lw=1.0))

# --- Sensibilidad global (muestreo del espacio completo) ------------------
N = 30_000
a = r.uniform(0, np.pi, N)
b = r.uniform(0, np.pi, N)
y = modelo(a, b)

# Índices de primer orden por binning (estimador de la varianza condicional)
def sobol_primer_orden(x, y, nbins=25):
    bins = np.linspace(x.min(), x.max(), nbins + 1)
    idx = np.digitize(x, bins) - 1
    medias = np.array([y[idx == k].mean() for k in range(nbins)
                       if np.any(idx == k)])
    return np.var(medias) / np.var(y)


S_a = sobol_primer_orden(a, y)
S_b = sobol_primer_orden(b, y)
ax2.bar(["$a$", "$b$"], [S_a, S_b], color=[C.blue, C.red], width=0.5)
ax2.set_ylabel("índice de Sobol de primer orden")
ax2.set_title("Global: los dos importan, y hay interacción")
ax2.text(0.5, max(S_a, S_b) * 0.75,
         f"$S_a$ = {S_a:.2f}\n$S_b$ = {S_b:.2f}\n"
         f"interacción = {1 - S_a - S_b:.2f}",
         fontsize=9.5, ha="center", color=C.ink,
         bbox=dict(boxstyle="round,pad=0.4", fc=C.light, ec=C.grey, lw=0.6))
ax2.set_ylim(0, max(S_a, S_b) * 1.35)

print(f"derivada parcial local en b: {(modelo(A0, B0+1e-6)-modelo(A0, B0))/1e-6:.2e}")
print(f"S_a = {S_a:.3f}, S_b = {S_b:.3f}, interacción = {1-S_a-S_b:.3f}")
plt.show()

**¿Ocurrió lo que esperabas? ¿Por qué?**

> 

**Juega:** cambia un parámetro de la celda anterior, vuelve a ejecutarla y anota el efecto.

> 


---

## Sobreajuste: el modelo que aprende el ruido.

Ajuste polinómico de grado creciente a pocos datos: error en entrenamiento, en
validación, y el comportamiento fuera del rango.

La figura responde: ¿cómo se detecta el sobreajuste sin conocer la verdad?

Ejecutar:  python fig_sobreajuste.py

*(script original: `codigo/fig_sobreajuste.py`)*

**Antes de ejecutar, escribe aquí qué esperas ver:**

> 


In [ ]:
import pathlib
import sys

import matplotlib.pyplot as plt
import numpy as np

from estilo_libro import C, rng, save, use_style  # noqa: E402

use_style()
r = rng(15)

SIGMA = 0.25


def verdad(x):
    return np.sin(1.6 * x) + 0.25 * x


x_ent = np.sort(r.uniform(0, 5, 14))
y_ent = verdad(x_ent) + r.normal(0, SIGMA, x_ent.size)
x_val = np.sort(r.uniform(0, 5, 200))
y_val = verdad(x_val) + r.normal(0, SIGMA, x_val.size)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10.4, 4.2))

xx = np.linspace(-0.3, 5.6, 500)
for grado, color, ancho in [(1, C.grey, 1.2), (4, C.green, 2.0),
                            (12, C.red, 1.6)]:
    c = np.polyfit(x_ent, y_ent, grado)
    ax1.plot(xx, np.polyval(c, xx), color=color, lw=ancho,
             label=f"grado {grado}")
ax1.plot(xx, verdad(xx), "--", color=C.ink, lw=1.6, label="verdad")
ax1.plot(x_ent, y_ent, "o", color=C.blue, ms=6, label="14 datos")
ax1.set_ylim(-2.6, 3.4), ax1.set_xlim(-0.3, 5.6)
ax1.set_xlabel("$x$"), ax1.set_ylabel("$y$")
ax1.set_title("El grado 12 pasa por todos los puntos")
ax1.legend(fontsize=8, loc="lower left")

grados = np.arange(0, 13)
err_ent, err_val = [], []
for g in grados:
    c = np.polyfit(x_ent, y_ent, g)
    err_ent.append(np.sqrt(np.mean((y_ent - np.polyval(c, x_ent))**2)))
    err_val.append(np.sqrt(np.mean((y_val - np.polyval(c, x_val))**2)))

ax2.semilogy(grados, err_ent, "o-", color=C.blue, ms=5, lw=1.6,
             label="error sobre los 14 datos usados")
ax2.semilogy(grados, err_val, "s-", color=C.red, ms=5, lw=1.6,
             label="error sobre datos nuevos")
ax2.axhline(SIGMA, color=C.ink, ls="--", lw=1.2)
ax2.text(0.2, SIGMA * 1.15, "ruido de medida", fontsize=8.4, color=C.ink)
g_opt = int(np.argmin(err_val))
ax2.axvline(g_opt, color=C.green, lw=1.2)
ax2.text(g_opt + 0.15, 3, f"óptimo: grado {g_opt}", fontsize=8.4, color=C.green)
ax2.set_xlabel("grado del polinomio"), ax2.set_ylabel("error rms")
ax2.set_title("El error de entrenamiento siempre baja")
ax2.legend(fontsize=8)

print(f"grado óptimo por validación: {g_opt}")
for g in (1, 4, 12):
    c = np.polyfit(x_ent, y_ent, g)
    print(f"grado {g:2d}: entrenamiento {np.sqrt(np.mean((y_ent-np.polyval(c,x_ent))**2)):.3f}, "
          f"validación {np.sqrt(np.mean((y_val-np.polyval(c,x_val))**2)):.3f}")
plt.show()

**¿Ocurrió lo que esperabas? ¿Por qué?**

> 

**Juega:** cambia un parámetro de la celda anterior, vuelve a ejecutarla y anota el efecto.

> 
